# 01 · Train and quantize the causal classifiers

Two public three-axis accelerometer datasets share one temporal CNN design. Models for the two datasets are trained independently. Subject-level splits are fixed before window extraction and remain fixed across the three training seeds. Only training inputs contribute to normalization and activation calibration.

Set `TINYML_NOTEBOOK_REBUILD=1` to download the original datasets if necessary, reproduce training, and compare the learned exports. Set `TINYML_DATA_DIR` to reuse local archives.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "study.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tinyml.config import DATASETS, RESULTS, SEEDS, run_dir

REBUILD = os.environ.get("TINYML_NOTEBOOK_REBUILD") == "1"
DATA = Path(os.environ.get("TINYML_DATA_DIR", ROOT / "data")).resolve()
WORKERS = os.environ.get("TINYML_NOTEBOOK_WORKERS", "2")
SCRATCH = ROOT / "build" / "notebooks"

def run(*args):
    command = [sys.executable, *map(str, args)]
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout + result.stderr)
        result.check_returncode()
    return result.stdout + result.stderr

print("Execution mode:", "full reproduction" if REBUILD else "inspect archived outputs")
print("Models:", len(DATASETS) * len(SEEDS), "| seeds:", ", ".join(map(str, SEEDS)))

Execution mode: full reproduction
Models: 6 | seeds: 13, 37, 73


In [2]:
if REBUILD:
    print(run("-m", "tinyml.data", "--data-dir", DATA), end="")
    print(run("-m", "tinyml.train", "--data-dir", DATA, "--results", SCRATCH,
              "--workers", WORKERS, "--threads", 1), end="")
    verified = 0
    for dataset in DATASETS:
        for seed in SEEDS:
            archive = run_dir(RESULTS, dataset, seed)
            rebuilt = run_dir(SCRATCH, dataset, seed)
            for name in ("fp32.npz", "int8.npz"):
                with np.load(archive / name) as expected, np.load(rebuilt / name) as actual:
                    assert set(expected.files) == set(actual.files)
                    for key in expected.files:
                        np.testing.assert_array_equal(actual[key], expected[key])
            verified += 1
    print(f"{verified} reproduced exports matched all archived FP32 and INT8 parameter arrays.")

wisdm: 1059 segments, 3,771,917 samples
  {'test': [1600, 1601, 1616, 1620, 1625, 1627, 1630, 1638, 1642, 1647], 'validation': [1605, 1610, 1614, 1624, 1632, 1635, 1649], 'train': [1602, 1603, 1604, 1606, 1607, 1608, 1609, 1611, 1612, 1613, 1615, 1617, 1618, 1619, 1621, 1622, 1623, 1626, 1628, 1629, 1631, 1633, 1634, 1636, 1637, 1639, 1640, 1641, 1643, 1644, 1645, 1646, 1648, 1650]}
  /workspace/scratch/cc21b0016654/.tinyml_work/data/wisdm_segments_v1.npz
mhealth: 126 segments, 343,195 samples
  {'test': [1, 4, 9], 'validation': [8, 10], 'train': [2, 3, 5, 6, 7]}
  /workspace/scratch/cc21b0016654/.tinyml_work/data/mhealth_segments_v1.npz


wisdm/73 epoch  1/18: val macro-F1=0.2679
wisdm/13 epoch  1/18: val macro-F1=0.2815
wisdm/37 epoch  1/18: val macro-F1=0.2455
wisdm/13 epoch  5/18: val macro-F1=0.4703
wisdm/73 epoch  5/18: val macro-F1=0.4740
wisdm/37 epoch  5/18: val macro-F1=0.4873
wisdm/13 epoch 10/18: val macro-F1=0.5204
wisdm/37 epoch 10/18: val macro-F1=0.5107
wisdm/73 epoch 10/18: val macro-F1=0.4951
wisdm/37 epoch 15/18: val macro-F1=0.5013
wisdm/13 epoch 15/18: val macro-F1=0.5237
wisdm/73 epoch 15/18: val macro-F1=0.5017
wisdm/37 epoch 18/18: val macro-F1=0.5104
Completed wisdm/37: FP32 0.426, INT8 0.416
mhealth/13 epoch  1/30: val macro-F1=0.0935
mhealth/13 epoch  5/30: val macro-F1=0.4055
wisdm/13 epoch 18/18: val macro-F1=0.5222
mhealth/13 epoch 10/30: val macro-F1=0.4361
Completed wisdm/13: FP32 0.431, INT8 0.418
mhealth/37 epoch  1/30: val macro-F1=0.0670
mhealth/13 epoch 15/30: val macro-F1=0.4441
wisdm/73 epoch 18/18: val macro-F1=0.5005
mhealth/37 epoch  5/30: val macro-F1=0.4118
mhealth/13 epoch 20/

6 reproduced exports matched all archived FP32 and INT8 parameter arrays.


In [3]:
from tinyml.quant import Int8Model

rows = []
for dataset in DATASETS:
    for seed in SEEDS:
        path = run_dir(RESULTS, dataset, seed)
        record = json.loads((path / "metrics.json").read_text())
        model = Int8Model.load(path / "int8.npz")
        assert not (set(record["subjects"]["train"]) & set(record["subjects"]["test"]))
        assert not (set(record["subjects"]["validation"]) & set(record["subjects"]["test"]))
        assert not (set(record["subjects"]["validation"]) & set(record["subjects"]["train"]))
        rows.append({
            "dataset": dataset.upper(), "seed": seed, "epoch": record["selected_epoch"],
            "training windows": record["windows"]["train"], "held-out windows": record["windows"]["test"],
            "FP32 accuracy (%)": 100 * record["fp32"]["accuracy"],
            "INT8 accuracy (%)": 100 * record["int8"]["accuracy"],
            "INT8 macro-F1 (%)": 100 * record["int8"]["macro_f1"],
            "INT8 weights": sum(x.weight.size for x in model.layers),
        })
display(pd.DataFrame(rows).round(2))

,dataset,seed,epoch,training windows,held-out windows,FP32 accuracy (%),INT8 accuracy (%),INT8 macro-F1 (%),INT8 weights
0,WISDM,13,16,13408,2668,43.15,41.85,40.99,1404
1,WISDM,37,16,13408,2668,42.55,41.59,41.52,1404
2,WISDM,73,13,13408,2668,40.70,40.29,40.08,1404
3,MHEALTH,13,20,2232,864,56.73,56.65,54.35,1308
4,MHEALTH,37,6,2232,864,46.61,45.54,40.65,1308
5,MHEALTH,73,19,2232,864,53.39,55.29,52.69,1308


The correctness target for repair is the clean INT8 model's score vector at each affected time. The exported networks are compact benchmark workloads. The held-out classification metrics above quantify their predictive quality independently of the correction mechanism.